# Q2 Pairs Trading Framework
###   *by Dr. W. Vera-Tudela*

## 1. Introduction & Objective

This is a custom pairs trading framework built from scratch in Jupyter Notebook with supporting functions in a separate Python file. The framework allows you to test a pair of stocks together and compare them against historical market data to evaluate their performance before risking real capital.

Objectives:
- Provide an evidence-based decision-making tool: Test your trading hypotheses with historical data.
- Allow strategy refinement: Identify strengths and weaknesses in your approach.
- Risk management: Understand potential drawdowns and volatility
- Performance comparison: Benchmark the pair against each stock independently and market indices
- Flexibility: Custom-built framework allows for complete control over testing parameters

This framework provides a solid foundation for quantitative trading strategy development and evaluation.

## 2. Data Pipeline

The structure of the data flow is as follows:

1. Data Management
    - Fetches and cleans real market data from Yahoo Finance
    - Handles historical price data for assets like AAPL and SPY
    - Stores processed data for efficient reuse  

2. Pair Checking and Signal Generation - stationary spread between two assets
    - Checks p-values for each asset and for both as a pair
    - Calculates the spread and z-value
    - Generates a signal to short/long each of the assets 

3. Pairs trading
    - Simulates portfolio evolution with precise cash and position tracking
    - Handles trade execution based on generated signals
    - Manages multiple positions and cash balance throughout the simulation  

4. Performance Analysis
    - Measures performance using multiple meaningful metrics:
        - Returns (absolute and relative)
        - Risk-adjusted metrics (Sharpe ratio)
        - Drawdown analysis (maximum drawdown)
    - Compares strategy performance against independent assets and benchmark indices  

5. Visualisation
    - Displays equity curves to track portfolio growth
    - Shows drawdown charts to visualise risk periods
    - Provides comparative performance visualisations against benchmarks  

This consolidated framework provides a complete pipeline from data acquisition through strategy testing to performance evaluation and visualisation.

### optimizer.py — all reusable logic:

fetch_returns() — yfinance pull → log returns  
compute_stats() — annualised μ and Σ (252-day scale)  
PortfolioOptimizer class with .min_variance(), .max_sharpe(), .efficient_frontier(), .random_portfolios()  
plot_frontier() — frontier line + Monte Carlo scatter (Sharpe-coloured) + Capital Market Line + annotated special portfolios  
plot_weights() — grouped bar chart with percentage labels  


### notebook.ipynb — 6-cell flow:

1. Pull data
2. Print μ/correlation matrix
3. Run all three optimisations
4. Frontier plot
5. Weight bars
6. Summary table with a key-takeaways markdown cell

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import seaborn as sns
import cvxpy

import sys
sys.path.append('../utils')   # path relative to the notebook
from F2_functions import fetch_returns, compute_stats, PortfolioOptimizer, plot_frontier, plot_weights

from common import fetch_data

## 3. Signal Generation

The first thing to do is to download the historical data for a given time period, then clean it to remove incomplete values and keep only the relevant columns.  
This is done by the function `fetch_data()`.

In [8]:
# Load data
period = 10
end = pd.Timestamp.today(tz="UTC").normalize()
start = end - pd.DateOffset(years=period)

returns = fetch_returns(['AAPL', 'NVDA'], start, end)

In [9]:
returns

Ticker,AAPL,NVDA
Date,,
2016-04-21,-0.010887,-0.001098
2016-04-22,-0.002741,-0.003852
2016-04-25,-0.005694,0.004951
2016-04-26,-0.006971,0.000823
2016-04-27,-0.064622,0.022497
...,...,...
2026-04-13,-0.004926,0.003598
2026-04-14,-0.001429,0.037327
2026-04-15,0.028940,0.011938


In [25]:
res = compute_stats(returns)

In [32]:
opt = PortfolioOptimizer(res['mu'], res['sigma'], res['tickers'])

In [34]:
plot_frontier(opt)

TypeError: plot_frontier() missing 2 required positional arguments: 'max_sharpe' and 'min_vol'

## 4. Pairs Trading Engine
Once the data and signals have been prepared, we move on to the pairs trading engine, the core of this notebook.  
Here, we implement a dual strategy:
1. Long a stock that should go up by buying as many shares as possible, and short a stock that should go down by borrowing an equivalent amount.
2. The previously generated signal tells us when to buy and when to sell. Also, depending on the signal sign (pos/neg) it tells us which stock to short and which to long.  

This is done by the function `run_pairs_trading()`.

In [ ]:
dataP = run_pairs_trading(dataV, dataM, signal, starting_capital)

In [ ]:
plt.figure(figsize=(24, 8))
colors = sns.color_palette("colorblind")

plt.subplot(1, 3, 1)
plt.plot(dataP.index,dataP['Long_Shares'],label='Long Shares',alpha=0.5)
plt.plot(dataP.index,dataP['Short_Shares'],label='Short Shares',alpha=0.5)
plt.title('Number of Shares')
plt.xlabel('Year')
plt.ylabel('Shares')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(dataP.index,dataP['Long_Leg'],label='Long Leg',alpha=0.5)
plt.plot(dataP.index,dataP['Short_Leg'],label='Short Leg',alpha=0.5)
plt.title('Leg Value')
plt.xlabel('Year')
plt.ylabel('Value')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(dataP.index,dataP['Total'],label='Total',alpha=0.75, color=colors[0])
plt.plot(dataP.index,dataP['Cash'],label='Cash', color=colors[1])
plt.plot(dataP.index,dataP['Portfolio_Value'],label='Portfolio', color=colors[2])
plt.title('Value')
plt.xlabel('Year')
plt.ylabel('Value')
plt.legend()

plt.show()

## 5. Performance Metrics
After the backtesting is finished and we have calculated our portfolio value and cash over time, we calculate the most relevant metrics to compare both strategies with each other and to the S&P500 index and to each asset independently.  
We calculate the total returns, Sharpe ratio, maximum drawdown, compounding annual growth rate (CAGR), and Calmar ratio as single metrics. In addition, we calculate the yearly returns.  
This is done by the function `compute_metrics()`.

In [ ]:
dataV, tableV, yV = compute_metrics(dataV,starting_capital)
dataM, tableM, yM = compute_metrics(dataM,starting_capital)
dataS, tableS, yS = compute_metrics(dataS,starting_capital)
dataP, tableP, yP = compute_metrics(dataP,starting_capital)

## 6. Results & Analysis
Finally, we show the results in the form of plots and a table for easier visualization.

The main results are shown as a triple plot:
1. The first one shows the equity curve on a daily basis, in which we compare the total value of our pairs trading strategy against buying and holding the same assets and the S&P500 index individually.
   - This is important as it shows how much our total portfolio value grows (or drops) in the same time period under different strategies.
2. The second plot shows the drawdowns on a daily basis, where we compare how much our portfolio has dropped in relation to the previous peaks for all three strategies.
    - This is important because it tells us how much our total value fluctuates compared to the other strategies.
3. The last plot shows the annual return, where as opposed to the total return metric, we show the return for each year.
   - This is important as it shows us if our portfolio has had some negative and positive years, which are not visible in the total return.

A table below shows the comparison of the main metrics for all strategies:
- Total Return
- Sharpe
- Max Drawdown
- CAGR
- Calmar

This is done by the function  `plot_performance()`.

In [ ]:
display(pd.concat([tableP, tableV, tableM, tableS], axis=1, keys=['Pairs', 'V', 'MA', 'SPY']))

plot_performance(dataV,dataM,dataS,dataP,yV,yM,yS,yP)

## 7. Conclusions

For this scenario: $10,000 starting capital, 10-year horizon on Visa (V) and Mastercard (MA), comparing a market-neutral pairs trading strategy against buy-and-hold V, MA, and SPY, we can draw the following conclusions:

- The pairs strategy delivers modest absolute returns but extraordinary capital preservation. Total return of 58% over 10 years (CAGR 4.7%) against buy-and-hold returns of 345% (V), 492% (MA), and 242% (SPY). The headline number looks poor — but the maximum drawdown of -2.1% is roughly 17x better than any alternative. The 2020 COVID crash that reduced all other strategies by 30-40% left the pairs portfolio virtually untouched.

- The strategy's value proposition is market neutrality, not return generation. By simultaneously longing one asset and shorting the other in dollar-neutral proportions, the portfolio is structurally insulated from broad market moves. Returns come entirely from the spread between V and MA — a relationship driven by relative performance, not market direction. This is a fundamentally different source of return from any directional strategy.

- V and MA were selected after rigorous statistical testing. Multiple candidate pairs (KO/PEP, XOM/CVX, MSFT/AAPL) failed the Engle-Granger cointegration test. V/MA passed with p = 0.0006 — the statistical relationship between them is robust, not assumed. This testing discipline is what separates genuine pairs trading from correlated-asset speculation.

- The Sharpe ratio understates the strategy's quality in isolation. Sharpe of 0.13 vs 0.45-0.52 for individual assets appears weak. But Sharpe penalises volatility — and this strategy has almost none (max drawdown -2.1%). The low Sharpe reflects low *absolute* returns, not poor risk management. In an institutional context, this strategy would be leveraged 3-4x, transforming 4.7% annual returns into 14-18% with still-manageable drawdown. That is how market-neutral strategies become commercially viable.

- Cointegration is not permanent. The V/MA relationship was validated over the full 10-year window, but the spread chart reveals increasing deviation in 2024-2025. Structural changes — regulatory divergence, international expansion differences, competitive shifts — can break cointegration permanently. A production implementation would test cointegration on a rolling basis and retire the strategy when the relationship degrades.

- Realized vs unrealized P&L are tracked separately. Metrics are calculated on realized cash (closed trades) rather than mark-to-market total value. This is a deliberate methodological choice — it more honestly reflects what the strategy actually delivered, since open position values fluctuate daily without generating real gains. A full implementation would report both.

The strategy is most suitable for capital-preservation mandates, market-neutral hedge fund structures, or as a low-correlation complement to a directional equity portfolio. Its true commercial value emerges only with leverage, which introduces margin requirements and borrow costs not modelled here.

## 8. Limitations & Next Steps

The three most important limitations of this model are the following:  
1. Zero borrow rate - under real conditions, an annualized fee would have to be taken into account, as well as a margin requirement.
2. Cointegration instability — the cointegration relationship between V and MA was tested on historical data, but it can break down due to regulatory changes, competitive shifts, or macroeconomic regime changes. The strategy has no mechanism to detect when the relationship has permanently broken and will continue generating false signals indefinitely.
3. Realized vs unrealized P&L — metrics are calculated on realized cash (closed trades) rather than mark-to-market total value. This understates intraday volatility but more honestly reflects what the strategy actually delivered. A full implementation would report both.

Some interesting next steps to go deeper on this project would be:
- Add transaction costs
- Test across multiple pairs
- Look for which asset characteristics make the strategy work best

Just to name a few.